# Vantare bicep-curl training, ONNX export, and live inference

## Goal

Train and evaluate the three-class personalized bicep-curl model from
`/content/DATASET`, export a verified ONNX model into that folder, and prove the
same preprocessing works for asynchronous live N2/N3/N4 sensor packets.

Run **Runtime → Run all** in Google Colab. No firmware is generated or changed.

## Setup

The notebook installs only the model-export/runtime packages that are not
normally included with Colab. The dataset path can be overridden with the
`VANTARE_DATASET_ROOT` environment variable for local verification.

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "skl2onnx": "skl2onnx>=1.18,<2",
    "onnx": "onnx>=1.17,<2",
    "onnxruntime": "onnxruntime>=1.20,<2",
}
missing = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("ONNX dependencies are ready.")

In [ ]:
from pathlib import Path
import json
import os
import platform
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)

import onnx
import onnxruntime as ort
import skl2onnx
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42
TARGET_HZ = 50
WINDOW_SECONDS = 2.0
STRIDE_SECONDS = 0.5
DATASET_ROOT = Path(os.environ.get("VANTARE_DATASET_ROOT", "/content/DATASET")).resolve()
OUTPUT_DIR = Path(os.environ.get("VANTARE_OUTPUT_DIR", str(DATASET_ROOT))).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset: {DATASET_ROOT}")
print(f"Outputs: {OUTPUT_DIR}")

## Context & Methods

- N2: wrist/distal forearm; N3: elbow region; N4: upper arm near shoulder.
- All six BNO85 and ICM45686 streams are interpolated to a shared 50 Hz clock.
- Each model input represents 2 seconds (100 synchronized samples).
- A new live result is produced every 0.5 seconds.
- Each window is summarized into 576 ordered `float32` features.
- Complete sessions are held out during evaluation; overlapping windows from one
  recording are never split across training and testing.

### Key assumptions and limitations

The data contains one person, one arm, two recordings per class, and no explicit
`rest`, `transition`, `shoulder_swing`, or `too_fast` class. Results are a Version
1 feasibility estimate, not evidence that the model generalizes to new users or
different sensor mounting. Do not enable automatic haptics during rest or setup
until those states are represented in later training data.

In [ ]:
"""Shared offline and live preprocessing for the Vantare bicep-curl model."""

from __future__ import annotations

from collections import deque
from dataclasses import dataclass
import json
from pathlib import Path
from typing import Iterable, Mapping

import numpy as np
import pandas as pd


NODES = (2, 3, 4)
SENSORS = ("BNO85", "ICM45686")
CLASS_NAMES = ("correct", "incomplete_range", "elbow_movement")
BNO_COLUMNS = (
    "quat_i", "quat_j", "quat_k", "quat_real",
    "linear_accel_x_mps2", "linear_accel_y_mps2", "linear_accel_z_mps2",
    "gravity_x_mps2", "gravity_y_mps2", "gravity_z_mps2",
    "gyro_x_radps", "gyro_y_radps", "gyro_z_radps",
)
ICM_COLUMNS = (
    "accel_x_g", "accel_y_g", "accel_z_g",
    "gyro_x_dps", "gyro_y_dps", "gyro_z_dps",
)
SUMMARY_STATISTICS = ("mean", "std", "min", "max", "range", "iqr", "rms", "mean_abs_diff")


def expected_sensor_columns(sensor: str) -> tuple[str, ...]:
    if sensor == "BNO85":
        return BNO_COLUMNS
    if sensor == "ICM45686":
        return ICM_COLUMNS
    raise ValueError(f"Unsupported sensor: {sensor}")


def validate_dataset(dataset_root: Path | str) -> dict:
    """Validate the labeled dataset contract and return its configuration."""
    root = Path(dataset_root)
    config_path = root / "dataset_config.json"
    if not config_path.is_file():
        raise FileNotFoundError(f"Missing dataset configuration: {config_path}")
    config = json.loads(config_path.read_text(encoding="utf-8"))
    sessions = config.get("sessions", [])
    if not sessions:
        raise ValueError("dataset_config.json contains no sessions")

    observed_ids = sorted({int(entry["class_id"]) for entry in sessions})
    if observed_ids != list(range(len(CLASS_NAMES))):
        raise ValueError(f"Expected class IDs 0..{len(CLASS_NAMES) - 1}; found {observed_ids}")

    for entry in sessions:
        session = int(entry["session"])
        folder = root / entry["folder"]
        if entry["label"] != CLASS_NAMES[int(entry["class_id"])]:
            raise ValueError(f"Class mapping mismatch for session {session}")
        for node in NODES:
            stem = f"R{session:04d}N{node}"
            required = (
                folder / f"{stem}_BNO85.csv",
                folder / f"{stem}_ICM45686.csv",
                folder / f"{stem}_metadata.json",
            )
            for path in required:
                if not path.is_file():
                    raise FileNotFoundError(f"Missing required dataset file: {path}")
    return config


def session_file(dataset_root: Path | str, entry: Mapping, node: int, sensor: str) -> Path:
    session = int(entry["session"])
    return Path(dataset_root) / str(entry["folder"]) / f"R{session:04d}N{node}_{sensor}.csv"


def load_numeric_stream(path: Path, columns: Iterable[str]) -> pd.DataFrame:
    columns = list(columns)
    frame = pd.read_csv(path, usecols=["timestamp_us", *columns])
    for column in ["timestamp_us", *columns]:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    if frame.isna().any().any():
        raise ValueError(f"Non-numeric or missing values in {path}")
    frame["time_s"] = frame.pop("timestamp_us") / 1_000_000.0
    frame = frame.drop_duplicates("time_s", keep="last").sort_values("time_s")
    if len(frame) < 2 or not frame["time_s"].is_monotonic_increasing:
        raise ValueError(f"Invalid timestamp series in {path}")
    return frame.set_index("time_s")


def interpolate_frame(frame: pd.DataFrame, grid: np.ndarray) -> pd.DataFrame:
    values = {
        column: np.interp(grid, frame.index.to_numpy(), frame[column].to_numpy())
        for column in frame.columns
    }
    return pd.DataFrame(values, index=grid)


def quaternion_angle_degrees(left: np.ndarray, right: np.ndarray) -> np.ndarray:
    left = left / np.clip(np.linalg.norm(left, axis=1, keepdims=True), 1e-9, None)
    right = right / np.clip(np.linalg.norm(right, axis=1, keepdims=True), 1e-9, None)
    dots = np.abs(np.sum(left * right, axis=1))
    return np.degrees(2 * np.arccos(np.clip(dots, 0, 1)))


def assemble_synchronized_frame(
    raw_streams: Mapping[tuple[int, str], pd.DataFrame], grid: np.ndarray
) -> pd.DataFrame:
    """Interpolate six streams and derive the 72 channels used by the model."""
    merged = pd.DataFrame(index=grid)
    for node in NODES:
        for sensor in SENSORS:
            frame = raw_streams[(node, sensor)]
            interpolated = interpolate_frame(frame, grid)
            if sensor == "BNO85":
                quaternion = interpolated[list(BNO_COLUMNS[:4])].to_numpy(copy=True)
                norms = np.linalg.norm(quaternion, axis=1, keepdims=True)
                if np.any(norms < 1e-9):
                    raise ValueError(f"Zero-length quaternion detected for N{node}")
                interpolated.loc[:, list(BNO_COLUMNS[:4])] = quaternion / norms
                prefix = f"n{node}_bno_"
            else:
                prefix = f"n{node}_icm_"
            interpolated.columns = [f"{prefix}{column}" for column in interpolated.columns]
            merged = merged.join(interpolated)

        magnitude_groups = {
            f"n{node}_bno_linear_accel_mag": [
                f"n{node}_bno_linear_accel_x_mps2", f"n{node}_bno_linear_accel_y_mps2",
                f"n{node}_bno_linear_accel_z_mps2",
            ],
            f"n{node}_bno_gyro_mag": [
                f"n{node}_bno_gyro_x_radps", f"n{node}_bno_gyro_y_radps", f"n{node}_bno_gyro_z_radps",
            ],
            f"n{node}_icm_accel_mag": [
                f"n{node}_icm_accel_x_g", f"n{node}_icm_accel_y_g", f"n{node}_icm_accel_z_g",
            ],
            f"n{node}_icm_gyro_mag": [
                f"n{node}_icm_gyro_x_dps", f"n{node}_icm_gyro_y_dps", f"n{node}_icm_gyro_z_dps",
            ],
        }
        for name, columns in magnitude_groups.items():
            merged[name] = np.linalg.norm(merged[columns].to_numpy(), axis=1)

    quaternion_suffixes = BNO_COLUMNS[:4]
    for left, right in ((2, 3), (3, 4), (2, 4)):
        left_q = merged[[f"n{left}_bno_{suffix}" for suffix in quaternion_suffixes]].to_numpy()
        right_q = merged[[f"n{right}_bno_{suffix}" for suffix in quaternion_suffixes]].to_numpy()
        merged[f"relative_angle_n{left}_n{right}_deg"] = quaternion_angle_degrees(left_q, right_q)
    merged.index.name = "time_s"
    return merged


def synchronize_session(
    dataset_root: Path | str, entry: Mapping, target_hz: int = 50
) -> pd.DataFrame:
    raw_streams = {}
    for node in NODES:
        for sensor in SENSORS:
            raw_streams[(node, sensor)] = load_numeric_stream(
                session_file(dataset_root, entry, node, sensor), expected_sensor_columns(sensor)
            )
    common_start = max(frame.index.min() for frame in raw_streams.values())
    common_end = min(frame.index.max() for frame in raw_streams.values())
    grid = np.arange(common_start, common_end, 1.0 / target_hz)
    if len(grid) < target_hz * 2:
        raise ValueError(f"Session {entry['session']} has less than two seconds of common data")
    return assemble_synchronized_frame(raw_streams, grid)


def build_feature_names(signal_columns: Iterable[str]) -> list[str]:
    return [f"{column}__{statistic}" for column in signal_columns for statistic in SUMMARY_STATISTICS]


def summarize_window(window: pd.DataFrame) -> dict[str, float]:
    values = window.to_numpy(dtype=float)
    features: dict[str, float] = {}
    for column_index, column in enumerate(window.columns):
        signal = values[:, column_index]
        statistics = (
            float(np.mean(signal)),
            float(np.std(signal)),
            float(np.min(signal)),
            float(np.max(signal)),
            float(np.ptp(signal)),
            float(np.percentile(signal, 75) - np.percentile(signal, 25)),
            float(np.sqrt(np.mean(signal ** 2))),
            float(np.mean(np.abs(np.diff(signal)))),
        )
        for statistic_name, value in zip(SUMMARY_STATISTICS, statistics):
            features[f"{column}__{statistic_name}"] = value
    return features


def window_session(
    frame: pd.DataFrame,
    session: int,
    class_id: int,
    target_hz: int = 50,
    window_seconds: float = 2.0,
    stride_seconds: float = 0.5,
) -> pd.DataFrame:
    window_samples = int(round(window_seconds * target_hz))
    stride_samples = int(round(stride_seconds * target_hz))
    rows = []
    for start in range(0, len(frame) - window_samples + 1, stride_samples):
        window = frame.iloc[start : start + window_samples]
        row = summarize_window(window)
        row.update(
            session=int(session),
            class_id=int(class_id),
            window_start_s=float(window.index[0]),
            window_end_s=float(window.index[-1]),
        )
        rows.append(row)
    return pd.DataFrame(rows)


@dataclass(frozen=True)
class LiveFeatureWindow:
    end_timestamp_us: int
    features: np.ndarray


class LiveWindowAssembler:
    """Turn asynchronous live sensor samples into offline-equivalent feature windows."""

    def __init__(self, target_hz: int = 50, window_seconds: float = 2.0, stride_seconds: float = 0.5):
        self.target_hz = int(target_hz)
        self.window_seconds = float(window_seconds)
        self.stride_seconds = float(stride_seconds)
        self.window_samples = int(round(self.target_hz * self.window_seconds))
        self.streams = {(node, sensor): deque() for node in NODES for sensor in SENSORS}
        self.last_emitted_end_s: float | None = None

    def push_sample(
        self,
        node: int,
        sensor: str,
        timestamp_us: int,
        values: Mapping[str, float],
    ) -> list[LiveFeatureWindow]:
        key = (int(node), sensor)
        if key not in self.streams:
            raise ValueError(f"Unsupported live stream: N{node} {sensor}")
        expected = expected_sensor_columns(sensor)
        missing = [column for column in expected if column not in values]
        if missing:
            raise ValueError(f"Missing {sensor} fields: {missing}")
        timestamp_s = int(timestamp_us) / 1_000_000.0
        stream = self.streams[key]
        if stream and timestamp_s <= stream[-1][0]:
            raise ValueError(f"Timestamps must increase for N{node} {sensor}")
        stream.append((timestamp_s, tuple(float(values[column]) for column in expected)))
        return self._emit_ready_windows()

    def _emit_ready_windows(self) -> list[LiveFeatureWindow]:
        if any(len(stream) < 2 for stream in self.streams.values()):
            return []
        common_end = min(stream[-1][0] for stream in self.streams.values())
        common_start = max(stream[0][0] for stream in self.streams.values())
        if common_end - common_start + 1e-9 < self.window_seconds:
            return []
        if self.last_emitted_end_s is not None and common_end - self.last_emitted_end_s + 1e-9 < self.stride_seconds:
            return []

        first_grid_time = common_end - (self.window_samples - 1) / self.target_hz
        if first_grid_time < common_start - 1e-9:
            return []
        grid = first_grid_time + np.arange(self.window_samples) / self.target_hz
        raw_frames = {}
        for (node, sensor), stream in self.streams.items():
            columns = expected_sensor_columns(sensor)
            data = list(stream)
            raw_frames[(node, sensor)] = pd.DataFrame(
                [row[1] for row in data], index=[row[0] for row in data], columns=columns
            )
        synchronized = assemble_synchronized_frame(raw_frames, grid)
        feature_values = np.fromiter(summarize_window(synchronized).values(), dtype=np.float32)
        self.last_emitted_end_s = common_end
        cutoff = common_end - self.window_seconds - self.stride_seconds
        for stream in self.streams.values():
            while len(stream) > 2 and stream[1][0] < cutoff:
                stream.popleft()
        return [LiveFeatureWindow(int(round(common_end * 1_000_000)), feature_values)]


@dataclass(frozen=True)
class HapticDecision:
    trigger_haptic: bool
    predicted_class_id: int
    predicted_label: str
    confidence: float
    consecutive_incorrect: int


class HapticGate:
    """Debounce incorrect predictions before requesting a haptic response."""

    def __init__(self, confidence_threshold: float = 0.70, required_consecutive: int = 2, cooldown_seconds: float = 2.0):
        self.confidence_threshold = float(confidence_threshold)
        self.required_consecutive = int(required_consecutive)
        self.cooldown_seconds = float(cooldown_seconds)
        self.consecutive_incorrect = 0
        self.last_trigger_s: float | None = None

    def update(self, class_id: int, probabilities: np.ndarray, timestamp_s: float) -> HapticDecision:
        class_id = int(class_id)
        probabilities = np.asarray(probabilities, dtype=float)
        if probabilities.shape != (len(CLASS_NAMES),):
            raise ValueError(f"Expected {len(CLASS_NAMES)} probabilities; got {probabilities.shape}")
        confidence = float(probabilities[class_id])
        confident_incorrect = class_id != 0 and confidence >= self.confidence_threshold
        self.consecutive_incorrect = self.consecutive_incorrect + 1 if confident_incorrect else 0
        cooldown_ready = self.last_trigger_s is None or timestamp_s - self.last_trigger_s >= self.cooldown_seconds
        trigger = self.consecutive_incorrect >= self.required_consecutive and cooldown_ready
        if trigger:
            self.last_trigger_s = float(timestamp_s)
        return HapticDecision(
            trigger_haptic=trigger,
            predicted_class_id=class_id,
            predicted_label=CLASS_NAMES[class_id],
            confidence=confidence,
            consecutive_incorrect=self.consecutive_incorrect,
        )


class RealtimeCurlClassifier:
    """Combine live windowing, model inference, and haptic decision stabilization."""

    def __init__(
        self,
        predictor,
        target_hz: int = 50,
        window_seconds: float = 2.0,
        stride_seconds: float = 0.5,
        confidence_threshold: float = 0.70,
        required_consecutive: int = 2,
        cooldown_seconds: float = 2.0,
    ):
        self.predictor = predictor
        self.assembler = LiveWindowAssembler(target_hz, window_seconds, stride_seconds)
        self.haptic_gate = HapticGate(confidence_threshold, required_consecutive, cooldown_seconds)

    def push_sample(
        self,
        node: int,
        sensor: str,
        timestamp_us: int,
        values: Mapping[str, float],
    ) -> list[dict]:
        results = []
        for window in self.assembler.push_sample(node, sensor, timestamp_us, values):
            labels, probabilities = self.predictor(window.features.reshape(1, -1).astype(np.float32))
            class_id = int(np.asarray(labels).reshape(-1)[0])
            class_probabilities = np.asarray(probabilities, dtype=np.float32).reshape(-1, len(CLASS_NAMES))[0]
            decision = self.haptic_gate.update(
                class_id, class_probabilities, window.end_timestamp_us / 1_000_000.0
            )
            results.append(
                {
                    "timestamp_us": window.end_timestamp_us,
                    "predicted_class_id": decision.predicted_class_id,
                    "predicted_label": decision.predicted_label,
                    "confidence": decision.confidence,
                    "probabilities": class_probabilities,
                    "trigger_haptic": decision.trigger_haptic,
                    "consecutive_incorrect": decision.consecutive_incorrect,
                }
            )
        return results

## Data

Validate the folder contract before any training work begins.

In [ ]:
config = validate_dataset(DATASET_ROOT)
session_entries = sorted(config["sessions"], key=lambda entry: int(entry["session"]))
folds = config.get("validation_folds", [])

assert len(session_entries) == 6, f"Expected 6 sessions, found {len(session_entries)}"
assert len(folds) == 2, f"Expected 2 session-held-out folds, found {len(folds)}"
print("Dataset contract passed:")
display(pd.DataFrame(session_entries))

### 1. Profile source files and sampling rates

In [ ]:
profile_rows = []
metadata_rows = []
for entry in session_entries:
    session = int(entry["session"])
    for node in NODES:
        metadata_path = DATASET_ROOT / entry["folder"] / f"R{session:04d}N{node}_metadata.json"
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        metadata_rows.append({
            "session": session,
            "node": node,
            "validated": bool(metadata.get("validated")),
            "bno_dropped": int(metadata.get("bno85_dropped_count", 0)),
            "icm_dropped": int(metadata.get("icm45686_dropped_count", 0)),
            "loss_flags": int(metadata.get("loss_flags", 0)),
        })
        for sensor in SENSORS:
            path = session_file(DATASET_ROOT, entry, node, sensor)
            frame = pd.read_csv(path)
            required = ["timestamp_us", *expected_sensor_columns(sensor)]
            missing_columns = sorted(set(required) - set(frame.columns))
            numeric = frame[required].apply(pd.to_numeric, errors="coerce") if not missing_columns else pd.DataFrame()
            timestamps = numeric["timestamp_us"] if not missing_columns else pd.Series(dtype=float)
            deltas = timestamps.diff().dropna()
            positive_deltas = deltas[deltas > 0]
            profile_rows.append({
                "session": session,
                "class": entry["label"],
                "node": node,
                "sensor": sensor,
                "rows": len(frame),
                "duration_s": (timestamps.iloc[-1] - timestamps.iloc[0]) / 1_000_000 if len(timestamps) else np.nan,
                "estimated_hz": 1_000_000 / positive_deltas.median() if len(positive_deltas) else np.nan,
                "missing_columns": ",".join(missing_columns),
                "missing_numeric": int(numeric.isna().sum().sum()) if not missing_columns else -1,
                "non_monotonic": int((deltas <= 0).sum()) if len(deltas) else -1,
            })

profiles = pd.DataFrame(profile_rows)
metadata_profile = pd.DataFrame(metadata_rows)
assert len(profiles) == 36
assert profiles["missing_columns"].eq("").all(), "Required sensor columns are missing"
assert profiles["missing_numeric"].eq(0).all(), "Sensor files contain missing/non-numeric model inputs"
assert profiles["non_monotonic"].eq(0).all(), "Sensor timestamps are not strictly increasing"
assert metadata_profile["validated"].all(), "At least one capture is not marked validated"

display(profiles.round({"duration_s": 2, "estimated_hz": 1}))
display(metadata_profile.groupby("session")[["bno_dropped", "icm_dropped", "loss_flags"]].sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for sensor, group in profiles.groupby("sensor"):
    axes[0].hist(group["estimated_hz"], bins=12, alpha=0.65, label=sensor)
axes[0].set(title="Observed source sampling rates", xlabel="Estimated samples/second", ylabel="Streams")
axes[0].legend()

duration_summary = profiles.groupby(["session", "class"], as_index=False)["duration_s"].min()
axes[1].bar(duration_summary["session"].astype(str), duration_summary["duration_s"], color="#2563eb")
axes[1].set(title="Common-duration lower bound by session", xlabel="Session", ylabel="Seconds")
axes[1].set_ylim(bottom=0)
plt.tight_layout()
plt.show()

### 2. Synchronize N2–N4 and inspect motion signals

In [ ]:
synchronized = {}
coverage_rows = []
for entry in session_entries:
    session = int(entry["session"])
    frame = synchronize_session(DATASET_ROOT, entry, target_hz=TARGET_HZ)
    synchronized[session] = frame
    coverage_rows.append({
        "session": session,
        "class": entry["label"],
        "samples_50hz": len(frame),
        "duration_s": frame.index[-1] - frame.index[0],
        "channels": frame.shape[1],
        "missing_cells": int(frame.isna().sum().sum()),
    })

coverage = pd.DataFrame(coverage_rows)
assert coverage["channels"].eq(72).all()
assert coverage["missing_cells"].eq(0).all()
display(coverage.round(2))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=False)
for axis, entry in zip(axes, session_entries[::2]):
    session = int(entry["session"])
    frame = synchronized[session]
    elapsed = frame.index.to_numpy() - frame.index[0]
    axis.plot(elapsed, frame["n2_bno_gyro_mag"], label="N2 wrist", alpha=0.85)
    axis.plot(elapsed, frame["n3_bno_gyro_mag"], label="N3 elbow", alpha=0.85)
    axis.plot(elapsed, frame["n4_bno_gyro_mag"], label="N4 shoulder", alpha=0.85)
    axis.set(title=f"Session {session}: {entry['label']}", ylabel="BNO gyro magnitude (rad/s)")
    axis.legend(loc="upper right", ncols=3)
axes[-1].set_xlabel("Elapsed time (seconds)")
plt.tight_layout()
plt.show()

### 3. Create offline/live-compatible feature windows

In [ ]:
window_frames = []
for entry in session_entries:
    session = int(entry["session"])
    window_frames.append(window_session(
        synchronized[session],
        session=session,
        class_id=int(entry["class_id"]),
        target_hz=TARGET_HZ,
        window_seconds=WINDOW_SECONDS,
        stride_seconds=STRIDE_SECONDS,
    ))
features = pd.concat(window_frames, ignore_index=True)
metadata_columns = {"session", "class_id", "window_start_s", "window_end_s"}
feature_names = [column for column in features.columns if column not in metadata_columns]

assert len(feature_names) == 576, f"Expected 576 features, found {len(feature_names)}"
assert feature_names == build_feature_names(synchronized[1].columns)
assert np.isfinite(features[feature_names].to_numpy()).all()

window_counts = features.groupby(["session", "class_id"]).size().rename("windows").reset_index()
window_counts["class"] = window_counts["class_id"].map(dict(enumerate(CLASS_NAMES)))
display(window_counts)

class_counts = features["class_id"].value_counts().sort_index()
plt.figure(figsize=(7, 4))
plt.bar([CLASS_NAMES[index] for index in class_counts.index], class_counts.values, color=["#16a34a", "#f59e0b", "#dc2626"])
plt.title("Training windows by class")
plt.ylabel("2-second windows")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Results

Fold A trains on sessions 1/3/5 and evaluates 2/4/6. Fold B reverses the
sessions. This is stronger than a random 90/10 window split because neighboring
overlapping windows from the same recording cannot leak into both sides.

In [ ]:
fold_metric_rows = []
prediction_frames = []
importance_frames = []

for fold in folds:
    train_sessions = [int(value) for value in fold["train_sessions"]]
    test_sessions = [int(value) for value in fold["test_sessions"]]
    train_mask = features["session"].isin(train_sessions)
    test_mask = features["session"].isin(test_sessions)
    X_train = features.loc[train_mask, feature_names].to_numpy(dtype=np.float32)
    y_train = features.loc[train_mask, "class_id"].to_numpy(dtype=np.int64)
    X_test = features.loc[test_mask, feature_names].to_numpy(dtype=np.float32)
    y_test = features.loc[test_mask, "class_id"].to_numpy(dtype=np.int64)

    model = RandomForestClassifier(
        n_estimators=500,
        max_depth=14,
        min_samples_leaf=3,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    predicted = model.predict(X_test)
    probabilities = model.predict_proba(X_test)
    fold_metric_rows.append({
        "fold": fold["name"],
        "train_sessions": ",".join(map(str, train_sessions)),
        "test_sessions": ",".join(map(str, test_sessions)),
        "test_windows": len(X_test),
        "accuracy": accuracy_score(y_test, predicted),
        "balanced_accuracy": balanced_accuracy_score(y_test, predicted),
        "macro_f1": f1_score(y_test, predicted, average="macro"),
    })
    predictions = features.loc[test_mask, ["session", "class_id", "window_start_s", "window_end_s"]].copy()
    predictions["fold"] = fold["name"]
    predictions["predicted_class_id"] = predicted
    for class_id in range(len(CLASS_NAMES)):
        predictions[f"probability_{CLASS_NAMES[class_id]}"] = probabilities[:, class_id]
    prediction_frames.append(predictions)
    importance_frames.append(pd.DataFrame({"fold": fold["name"], "feature": feature_names, "importance": model.feature_importances_}))

fold_metrics = pd.DataFrame(fold_metric_rows)
held_out_predictions = pd.concat(prediction_frames, ignore_index=True)
feature_importances = pd.concat(importance_frames, ignore_index=True)
display(fold_metrics.round(3))
display(fold_metrics[["accuracy", "balanced_accuracy", "macro_f1"]].mean().to_frame("mean").round(3))

In [ ]:
metric_columns = ["accuracy", "balanced_accuracy", "macro_f1"]
figure, axis = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(fold_metrics))
width = 0.24
for offset, metric in enumerate(metric_columns):
    axis.bar(x + (offset - 1) * width, fold_metrics[metric], width, label=metric.replace("_", " "))
axis.set_xticks(x, [f"Fold {name}" for name in fold_metrics["fold"]])
axis.set_ylim(0, 1.05)
axis.set_ylabel("Score")
axis.set_title("Complete-session held-out performance")
axis.legend()
plt.tight_layout()
plt.show()

In [ ]:
y_true = held_out_predictions["class_id"].to_numpy()
y_pred = held_out_predictions["predicted_class_id"].to_numpy()
count_matrix = confusion_matrix(y_true, y_pred, labels=range(len(CLASS_NAMES)))
normalized_matrix = confusion_matrix(y_true, y_pred, labels=range(len(CLASS_NAMES)), normalize="true")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for axis, matrix, title, value_format in [
    (axes[0], count_matrix, "Held-out confusion matrix: counts", "d"),
    (axes[1], normalized_matrix, "Held-out confusion matrix: row-normalized", ".2f"),
]:
    image = axis.imshow(matrix, cmap="Blues", vmin=0, vmax=None if value_format == "d" else 1)
    axis.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=25, ha="right")
    axis.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
    axis.set(xlabel="Predicted", ylabel="Actual", title=title)
    for row in range(len(CLASS_NAMES)):
        for column in range(len(CLASS_NAMES)):
            axis.text(column, row, format(matrix[row, column], value_format), ha="center", va="center")
    fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=range(len(CLASS_NAMES)), zero_division=0
)
per_class_metrics = pd.DataFrame({
    "class": CLASS_NAMES,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "held_out_windows": support,
})
display(per_class_metrics.round(3))

fig, axis = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(CLASS_NAMES))
for offset, metric in enumerate(["precision", "recall", "f1"]):
    axis.bar(x + (offset - 1) * 0.24, per_class_metrics[metric], 0.24, label=metric)
axis.set_xticks(x, CLASS_NAMES, rotation=15)
axis.set_ylim(0, 1.05)
axis.set_ylabel("Score")
axis.set_title("Per-class held-out performance")
axis.legend()
plt.tight_layout()
plt.show()

In [ ]:
top_features = (
    feature_importances.groupby("feature", as_index=False)["importance"].mean()
    .sort_values("importance", ascending=False)
    .head(25)
)
display(top_features)
plt.figure(figsize=(10, 7))
ordered = top_features.sort_values("importance")
plt.barh(ordered["feature"], ordered["importance"], color="#2563eb")
plt.xlabel("Mean importance across held-out folds")
plt.title("Top 25 model features")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 10), sharey=True)
for axis, (session, group) in zip(axes.flat, held_out_predictions.groupby("session")):
    group = group.sort_values("window_end_s")
    elapsed = group["window_end_s"] - group["window_end_s"].min()
    for class_name in CLASS_NAMES:
        axis.plot(elapsed, group[f"probability_{class_name}"], label=class_name)
    actual = CLASS_NAMES[int(group["class_id"].iloc[0])]
    axis.set(title=f"Held-out session {session} (actual: {actual})", xlabel="Elapsed seconds", ylabel="Probability", ylim=(0, 1.02))
axes[0, 0].legend(ncols=3, fontsize=8)
plt.tight_layout()
plt.show()

## Train final model and export ONNX

The evaluation above is preserved first. The deployable model is then retrained
on every available window. Consequently, it has more training data but no
remaining independent test session; use the held-out results above for the
honest Version 1 estimate.

In [ ]:
X_all = features[feature_names].to_numpy(dtype=np.float32)
y_all = features["class_id"].to_numpy(dtype=np.int64)
final_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=14,
    min_samples_leaf=3,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
final_model.fit(X_all, y_all)

onnx_model = convert_sklearn(
    final_model,
    name="vantare_bicep_curl_v1",
    initial_types=[("features", FloatTensorType([None, len(feature_names)]))],
    target_opset={"": 18, "ai.onnx.ml": 3},
    options={id(final_model): {"zipmap": False}},
)
onnx_path = OUTPUT_DIR / "vantare_bicep_curl_v1.onnx"
onnx_path.write_bytes(onnx_model.SerializeToString())
onnx.checker.check_model(onnx.load(onnx_path))

onnx_session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
input_info = onnx_session.get_inputs()[0]
output_info = onnx_session.get_outputs()
input_name = input_info.name
output_names = [item.name for item in output_info]

parity_batch = X_all[: min(128, len(X_all))]
sklearn_labels = final_model.predict(parity_batch)
sklearn_probabilities = final_model.predict_proba(parity_batch)
onnx_labels, onnx_probabilities = onnx_session.run(None, {input_name: parity_batch})
label_parity = float(np.mean(np.asarray(onnx_labels).reshape(-1) == sklearn_labels))
maximum_probability_difference = float(np.max(np.abs(np.asarray(onnx_probabilities) - sklearn_probabilities)))
assert label_parity == 1.0, "ONNX labels differ from scikit-learn"
assert maximum_probability_difference < 1e-5, "ONNX probabilities differ beyond tolerance"

print(f"Saved: {onnx_path}")
print(f"ONNX input:  {input_name} {input_info.type} {input_info.shape}")
for item in output_info:
    print(f"ONNX output: {item.name} {item.type} {item.shape}")
print(f"Label parity: {label_parity:.3f}")
print(f"Maximum probability difference: {maximum_probability_difference:.8f}")

In [ ]:
classification_report_data = classification_report(
    y_true,
    y_pred,
    labels=range(len(CLASS_NAMES)),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
metrics_artifact = {
    "evaluation_method": "two complete-session-held-out folds",
    "folds": json.loads(fold_metrics.to_json(orient="records")),
    "mean_metrics": {column: float(fold_metrics[column].mean()) for column in ["accuracy", "balanced_accuracy", "macro_f1"]},
    "classification_report": classification_report_data,
    "confusion_matrix_counts": count_matrix.tolist(),
    "confusion_matrix_normalized": normalized_matrix.tolist(),
    "onnx_parity": {
        "samples_checked": len(parity_batch),
        "label_agreement": label_parity,
        "maximum_probability_difference": maximum_probability_difference,
    },
}

contract = {
    "model_name": "vantare_bicep_curl_v1",
    "task": "personalized_bicep_curl_form_classification",
    "class_mapping": {str(index): name for index, name in enumerate(CLASS_NAMES)},
    "nodes": {"2": "wrist_distal_forearm", "3": "elbow_region", "4": "upper_arm_near_shoulder"},
    "live_sensor_packet_contract": {
        "common_fields": {"node": "integer 2, 3, or 4", "sensor": "BNO85 or ICM45686", "timestamp_us": "monotonic integer microseconds per stream"},
        "BNO85_values": list(BNO_COLUMNS),
        "ICM45686_values": list(ICM_COLUMNS),
    },
    "preprocessing": {
        "target_hz": TARGET_HZ,
        "window_seconds": WINDOW_SECONDS,
        "window_samples": int(TARGET_HZ * WINDOW_SECONDS),
        "stride_seconds": STRIDE_SECONDS,
        "raw_synchronized_channels": 72,
        "feature_count": len(feature_names),
        "feature_statistics": list(SUMMARY_STATISTICS),
        "quaternion_handling": "linear interpolation followed by unit normalization",
        "live_warmup_seconds": WINDOW_SECONDS,
        "prediction_update_seconds": STRIDE_SECONDS,
    },
    "onnx": {
        "file": onnx_path.name,
        "input": {"name": input_name, "dtype": "float32", "shape": ["batch_size", len(feature_names)]},
        "outputs": [
            {"name": output_info[0].name, "meaning": "predicted_class_id", "dtype": output_info[0].type, "shape": output_info[0].shape},
            {"name": output_info[1].name, "meaning": "class_probabilities", "dtype": output_info[1].type, "shape": output_info[1].shape, "column_order": list(CLASS_NAMES)},
        ],
    },
    "recommended_haptic_gate_v1": {
        "incorrect_classes": [1, 2],
        "minimum_probability": 0.70,
        "consecutive_windows": 2,
        "cooldown_seconds": 2.0,
        "minimum_time_before_first_possible_trigger_seconds": WINDOW_SECONDS + STRIDE_SECONDS,
    },
    "scope_warning": "Personalized feasibility model. No rest/transition class and no evidence for new users or changed mounting.",
    "software_versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "skl2onnx": skl2onnx.__version__,
        "onnx": onnx.__version__,
        "onnxruntime": ort.__version__,
    },
}

(OUTPUT_DIR / "model_contract.json").write_text(json.dumps(contract, indent=2), encoding="utf-8")
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(metrics_artifact, indent=2), encoding="utf-8")
(OUTPUT_DIR / "feature_names.json").write_text(json.dumps(feature_names, indent=2), encoding="utf-8")

In [ ]:
PIPELINE_SOURCE = '"""Shared offline and live preprocessing for the Vantare bicep-curl model."""\n\nfrom __future__ import annotations\n\nfrom collections import deque\nfrom dataclasses import dataclass\nimport json\nfrom pathlib import Path\nfrom typing import Iterable, Mapping\n\nimport numpy as np\nimport pandas as pd\n\n\nNODES = (2, 3, 4)\nSENSORS = ("BNO85", "ICM45686")\nCLASS_NAMES = ("correct", "incomplete_range", "elbow_movement")\nBNO_COLUMNS = (\n    "quat_i", "quat_j", "quat_k", "quat_real",\n    "linear_accel_x_mps2", "linear_accel_y_mps2", "linear_accel_z_mps2",\n    "gravity_x_mps2", "gravity_y_mps2", "gravity_z_mps2",\n    "gyro_x_radps", "gyro_y_radps", "gyro_z_radps",\n)\nICM_COLUMNS = (\n    "accel_x_g", "accel_y_g", "accel_z_g",\n    "gyro_x_dps", "gyro_y_dps", "gyro_z_dps",\n)\nSUMMARY_STATISTICS = ("mean", "std", "min", "max", "range", "iqr", "rms", "mean_abs_diff")\n\n\ndef expected_sensor_columns(sensor: str) -> tuple[str, ...]:\n    if sensor == "BNO85":\n        return BNO_COLUMNS\n    if sensor == "ICM45686":\n        return ICM_COLUMNS\n    raise ValueError(f"Unsupported sensor: {sensor}")\n\n\ndef validate_dataset(dataset_root: Path | str) -> dict:\n    """Validate the labeled dataset contract and return its configuration."""\n    root = Path(dataset_root)\n    config_path = root / "dataset_config.json"\n    if not config_path.is_file():\n        raise FileNotFoundError(f"Missing dataset configuration: {config_path}")\n    config = json.loads(config_path.read_text(encoding="utf-8"))\n    sessions = config.get("sessions", [])\n    if not sessions:\n        raise ValueError("dataset_config.json contains no sessions")\n\n    observed_ids = sorted({int(entry["class_id"]) for entry in sessions})\n    if observed_ids != list(range(len(CLASS_NAMES))):\n        raise ValueError(f"Expected class IDs 0..{len(CLASS_NAMES) - 1}; found {observed_ids}")\n\n    for entry in sessions:\n        session = int(entry["session"])\n        folder = root / entry["folder"]\n        if entry["label"] != CLASS_NAMES[int(entry["class_id"])]:\n            raise ValueError(f"Class mapping mismatch for session {session}")\n        for node in NODES:\n            stem = f"R{session:04d}N{node}"\n            required = (\n                folder / f"{stem}_BNO85.csv",\n                folder / f"{stem}_ICM45686.csv",\n                folder / f"{stem}_metadata.json",\n            )\n            for path in required:\n                if not path.is_file():\n                    raise FileNotFoundError(f"Missing required dataset file: {path}")\n    return config\n\n\ndef session_file(dataset_root: Path | str, entry: Mapping, node: int, sensor: str) -> Path:\n    session = int(entry["session"])\n    return Path(dataset_root) / str(entry["folder"]) / f"R{session:04d}N{node}_{sensor}.csv"\n\n\ndef load_numeric_stream(path: Path, columns: Iterable[str]) -> pd.DataFrame:\n    columns = list(columns)\n    frame = pd.read_csv(path, usecols=["timestamp_us", *columns])\n    for column in ["timestamp_us", *columns]:\n        frame[column] = pd.to_numeric(frame[column], errors="coerce")\n    if frame.isna().any().any():\n        raise ValueError(f"Non-numeric or missing values in {path}")\n    frame["time_s"] = frame.pop("timestamp_us") / 1_000_000.0\n    frame = frame.drop_duplicates("time_s", keep="last").sort_values("time_s")\n    if len(frame) < 2 or not frame["time_s"].is_monotonic_increasing:\n        raise ValueError(f"Invalid timestamp series in {path}")\n    return frame.set_index("time_s")\n\n\ndef interpolate_frame(frame: pd.DataFrame, grid: np.ndarray) -> pd.DataFrame:\n    values = {\n        column: np.interp(grid, frame.index.to_numpy(), frame[column].to_numpy())\n        for column in frame.columns\n    }\n    return pd.DataFrame(values, index=grid)\n\n\ndef quaternion_angle_degrees(left: np.ndarray, right: np.ndarray) -> np.ndarray:\n    left = left / np.clip(np.linalg.norm(left, axis=1, keepdims=True), 1e-9, None)\n    right = right / np.clip(np.linalg.norm(right, axis=1, keepdims=True), 1e-9, None)\n    dots = np.abs(np.sum(left * right, axis=1))\n    return np.degrees(2 * np.arccos(np.clip(dots, 0, 1)))\n\n\ndef assemble_synchronized_frame(\n    raw_streams: Mapping[tuple[int, str], pd.DataFrame], grid: np.ndarray\n) -> pd.DataFrame:\n    """Interpolate six streams and derive the 72 channels used by the model."""\n    merged = pd.DataFrame(index=grid)\n    for node in NODES:\n        for sensor in SENSORS:\n            frame = raw_streams[(node, sensor)]\n            interpolated = interpolate_frame(frame, grid)\n            if sensor == "BNO85":\n                quaternion = interpolated[list(BNO_COLUMNS[:4])].to_numpy(copy=True)\n                norms = np.linalg.norm(quaternion, axis=1, keepdims=True)\n                if np.any(norms < 1e-9):\n                    raise ValueError(f"Zero-length quaternion detected for N{node}")\n                interpolated.loc[:, list(BNO_COLUMNS[:4])] = quaternion / norms\n                prefix = f"n{node}_bno_"\n            else:\n                prefix = f"n{node}_icm_"\n            interpolated.columns = [f"{prefix}{column}" for column in interpolated.columns]\n            merged = merged.join(interpolated)\n\n        magnitude_groups = {\n            f"n{node}_bno_linear_accel_mag": [\n                f"n{node}_bno_linear_accel_x_mps2", f"n{node}_bno_linear_accel_y_mps2",\n                f"n{node}_bno_linear_accel_z_mps2",\n            ],\n            f"n{node}_bno_gyro_mag": [\n                f"n{node}_bno_gyro_x_radps", f"n{node}_bno_gyro_y_radps", f"n{node}_bno_gyro_z_radps",\n            ],\n            f"n{node}_icm_accel_mag": [\n                f"n{node}_icm_accel_x_g", f"n{node}_icm_accel_y_g", f"n{node}_icm_accel_z_g",\n            ],\n            f"n{node}_icm_gyro_mag": [\n                f"n{node}_icm_gyro_x_dps", f"n{node}_icm_gyro_y_dps", f"n{node}_icm_gyro_z_dps",\n            ],\n        }\n        for name, columns in magnitude_groups.items():\n            merged[name] = np.linalg.norm(merged[columns].to_numpy(), axis=1)\n\n    quaternion_suffixes = BNO_COLUMNS[:4]\n    for left, right in ((2, 3), (3, 4), (2, 4)):\n        left_q = merged[[f"n{left}_bno_{suffix}" for suffix in quaternion_suffixes]].to_numpy()\n        right_q = merged[[f"n{right}_bno_{suffix}" for suffix in quaternion_suffixes]].to_numpy()\n        merged[f"relative_angle_n{left}_n{right}_deg"] = quaternion_angle_degrees(left_q, right_q)\n    merged.index.name = "time_s"\n    return merged\n\n\ndef synchronize_session(\n    dataset_root: Path | str, entry: Mapping, target_hz: int = 50\n) -> pd.DataFrame:\n    raw_streams = {}\n    for node in NODES:\n        for sensor in SENSORS:\n            raw_streams[(node, sensor)] = load_numeric_stream(\n                session_file(dataset_root, entry, node, sensor), expected_sensor_columns(sensor)\n            )\n    common_start = max(frame.index.min() for frame in raw_streams.values())\n    common_end = min(frame.index.max() for frame in raw_streams.values())\n    grid = np.arange(common_start, common_end, 1.0 / target_hz)\n    if len(grid) < target_hz * 2:\n        raise ValueError(f"Session {entry[\'session\']} has less than two seconds of common data")\n    return assemble_synchronized_frame(raw_streams, grid)\n\n\ndef build_feature_names(signal_columns: Iterable[str]) -> list[str]:\n    return [f"{column}__{statistic}" for column in signal_columns for statistic in SUMMARY_STATISTICS]\n\n\ndef summarize_window(window: pd.DataFrame) -> dict[str, float]:\n    values = window.to_numpy(dtype=float)\n    features: dict[str, float] = {}\n    for column_index, column in enumerate(window.columns):\n        signal = values[:, column_index]\n        statistics = (\n            float(np.mean(signal)),\n            float(np.std(signal)),\n            float(np.min(signal)),\n            float(np.max(signal)),\n            float(np.ptp(signal)),\n            float(np.percentile(signal, 75) - np.percentile(signal, 25)),\n            float(np.sqrt(np.mean(signal ** 2))),\n            float(np.mean(np.abs(np.diff(signal)))),\n        )\n        for statistic_name, value in zip(SUMMARY_STATISTICS, statistics):\n            features[f"{column}__{statistic_name}"] = value\n    return features\n\n\ndef window_session(\n    frame: pd.DataFrame,\n    session: int,\n    class_id: int,\n    target_hz: int = 50,\n    window_seconds: float = 2.0,\n    stride_seconds: float = 0.5,\n) -> pd.DataFrame:\n    window_samples = int(round(window_seconds * target_hz))\n    stride_samples = int(round(stride_seconds * target_hz))\n    rows = []\n    for start in range(0, len(frame) - window_samples + 1, stride_samples):\n        window = frame.iloc[start : start + window_samples]\n        row = summarize_window(window)\n        row.update(\n            session=int(session),\n            class_id=int(class_id),\n            window_start_s=float(window.index[0]),\n            window_end_s=float(window.index[-1]),\n        )\n        rows.append(row)\n    return pd.DataFrame(rows)\n\n\n@dataclass(frozen=True)\nclass LiveFeatureWindow:\n    end_timestamp_us: int\n    features: np.ndarray\n\n\nclass LiveWindowAssembler:\n    """Turn asynchronous live sensor samples into offline-equivalent feature windows."""\n\n    def __init__(self, target_hz: int = 50, window_seconds: float = 2.0, stride_seconds: float = 0.5):\n        self.target_hz = int(target_hz)\n        self.window_seconds = float(window_seconds)\n        self.stride_seconds = float(stride_seconds)\n        self.window_samples = int(round(self.target_hz * self.window_seconds))\n        self.streams = {(node, sensor): deque() for node in NODES for sensor in SENSORS}\n        self.last_emitted_end_s: float | None = None\n\n    def push_sample(\n        self,\n        node: int,\n        sensor: str,\n        timestamp_us: int,\n        values: Mapping[str, float],\n    ) -> list[LiveFeatureWindow]:\n        key = (int(node), sensor)\n        if key not in self.streams:\n            raise ValueError(f"Unsupported live stream: N{node} {sensor}")\n        expected = expected_sensor_columns(sensor)\n        missing = [column for column in expected if column not in values]\n        if missing:\n            raise ValueError(f"Missing {sensor} fields: {missing}")\n        timestamp_s = int(timestamp_us) / 1_000_000.0\n        stream = self.streams[key]\n        if stream and timestamp_s <= stream[-1][0]:\n            raise ValueError(f"Timestamps must increase for N{node} {sensor}")\n        stream.append((timestamp_s, tuple(float(values[column]) for column in expected)))\n        return self._emit_ready_windows()\n\n    def _emit_ready_windows(self) -> list[LiveFeatureWindow]:\n        if any(len(stream) < 2 for stream in self.streams.values()):\n            return []\n        common_end = min(stream[-1][0] for stream in self.streams.values())\n        common_start = max(stream[0][0] for stream in self.streams.values())\n        if common_end - common_start + 1e-9 < self.window_seconds:\n            return []\n        if self.last_emitted_end_s is not None and common_end - self.last_emitted_end_s + 1e-9 < self.stride_seconds:\n            return []\n\n        first_grid_time = common_end - (self.window_samples - 1) / self.target_hz\n        if first_grid_time < common_start - 1e-9:\n            return []\n        grid = first_grid_time + np.arange(self.window_samples) / self.target_hz\n        raw_frames = {}\n        for (node, sensor), stream in self.streams.items():\n            columns = expected_sensor_columns(sensor)\n            data = list(stream)\n            raw_frames[(node, sensor)] = pd.DataFrame(\n                [row[1] for row in data], index=[row[0] for row in data], columns=columns\n            )\n        synchronized = assemble_synchronized_frame(raw_frames, grid)\n        feature_values = np.fromiter(summarize_window(synchronized).values(), dtype=np.float32)\n        self.last_emitted_end_s = common_end\n        cutoff = common_end - self.window_seconds - self.stride_seconds\n        for stream in self.streams.values():\n            while len(stream) > 2 and stream[1][0] < cutoff:\n                stream.popleft()\n        return [LiveFeatureWindow(int(round(common_end * 1_000_000)), feature_values)]\n\n\n@dataclass(frozen=True)\nclass HapticDecision:\n    trigger_haptic: bool\n    predicted_class_id: int\n    predicted_label: str\n    confidence: float\n    consecutive_incorrect: int\n\n\nclass HapticGate:\n    """Debounce incorrect predictions before requesting a haptic response."""\n\n    def __init__(self, confidence_threshold: float = 0.70, required_consecutive: int = 2, cooldown_seconds: float = 2.0):\n        self.confidence_threshold = float(confidence_threshold)\n        self.required_consecutive = int(required_consecutive)\n        self.cooldown_seconds = float(cooldown_seconds)\n        self.consecutive_incorrect = 0\n        self.last_trigger_s: float | None = None\n\n    def update(self, class_id: int, probabilities: np.ndarray, timestamp_s: float) -> HapticDecision:\n        class_id = int(class_id)\n        probabilities = np.asarray(probabilities, dtype=float)\n        if probabilities.shape != (len(CLASS_NAMES),):\n            raise ValueError(f"Expected {len(CLASS_NAMES)} probabilities; got {probabilities.shape}")\n        confidence = float(probabilities[class_id])\n        confident_incorrect = class_id != 0 and confidence >= self.confidence_threshold\n        self.consecutive_incorrect = self.consecutive_incorrect + 1 if confident_incorrect else 0\n        cooldown_ready = self.last_trigger_s is None or timestamp_s - self.last_trigger_s >= self.cooldown_seconds\n        trigger = self.consecutive_incorrect >= self.required_consecutive and cooldown_ready\n        if trigger:\n            self.last_trigger_s = float(timestamp_s)\n        return HapticDecision(\n            trigger_haptic=trigger,\n            predicted_class_id=class_id,\n            predicted_label=CLASS_NAMES[class_id],\n            confidence=confidence,\n            consecutive_incorrect=self.consecutive_incorrect,\n        )\n\n\nclass RealtimeCurlClassifier:\n    """Combine live windowing, model inference, and haptic decision stabilization."""\n\n    def __init__(\n        self,\n        predictor,\n        target_hz: int = 50,\n        window_seconds: float = 2.0,\n        stride_seconds: float = 0.5,\n        confidence_threshold: float = 0.70,\n        required_consecutive: int = 2,\n        cooldown_seconds: float = 2.0,\n    ):\n        self.predictor = predictor\n        self.assembler = LiveWindowAssembler(target_hz, window_seconds, stride_seconds)\n        self.haptic_gate = HapticGate(confidence_threshold, required_consecutive, cooldown_seconds)\n\n    def push_sample(\n        self,\n        node: int,\n        sensor: str,\n        timestamp_us: int,\n        values: Mapping[str, float],\n    ) -> list[dict]:\n        results = []\n        for window in self.assembler.push_sample(node, sensor, timestamp_us, values):\n            labels, probabilities = self.predictor(window.features.reshape(1, -1).astype(np.float32))\n            class_id = int(np.asarray(labels).reshape(-1)[0])\n            class_probabilities = np.asarray(probabilities, dtype=np.float32).reshape(-1, len(CLASS_NAMES))[0]\n            decision = self.haptic_gate.update(\n                class_id, class_probabilities, window.end_timestamp_us / 1_000_000.0\n            )\n            results.append(\n                {\n                    "timestamp_us": window.end_timestamp_us,\n                    "predicted_class_id": decision.predicted_class_id,\n                    "predicted_label": decision.predicted_label,\n                    "confidence": decision.confidence,\n                    "probabilities": class_probabilities,\n                    "trigger_haptic": decision.trigger_haptic,\n                    "consecutive_incorrect": decision.consecutive_incorrect,\n                }\n            )\n        return results\n'
runtime_module_path = OUTPUT_DIR / "vantare_bicep_curl_pipeline.py"
runtime_module_path.write_text(PIPELINE_SOURCE, encoding="utf-8")
print(f"Saved live preprocessing module: {runtime_module_path}")

## Live ONNX integration smoke test

This replays one recorded session through the exact public live API. Packets are
sorted by sensor timestamp and processed as quickly as possible; no artificial
sleep is used. In the application, call `push_sample(...)` whenever a live N2,
N3, or N4 BNO85/ICM45686 packet arrives. Most calls return an empty list. Once a
complete synchronized window is available, the returned list contains the
prediction, all three probabilities, and `trigger_haptic`.

This is an integration check on training data—not an additional accuracy test.

In [ ]:
def onnx_predict(feature_batch):
    return onnx_session.run(None, {input_name: np.asarray(feature_batch, dtype=np.float32)})


def replay_recording_as_live(entry):
    events = []
    session = int(entry["session"])
    for node in NODES:
        for sensor in SENSORS:
            columns = list(expected_sensor_columns(sensor))
            frame = pd.read_csv(session_file(DATASET_ROOT, entry, node, sensor), usecols=["timestamp_us", *columns])
            for row in frame.itertuples(index=False, name=None):
                timestamp_us = int(row[0])
                values = dict(zip(columns, row[1:]))
                events.append((timestamp_us, node, sensor, values))
    events.sort(key=lambda event: event[0])

    runtime = RealtimeCurlClassifier(
        predictor=onnx_predict,
        target_hz=TARGET_HZ,
        window_seconds=WINDOW_SECONDS,
        stride_seconds=STRIDE_SECONDS,
        confidence_threshold=0.70,
        required_consecutive=2,
        cooldown_seconds=2.0,
    )
    results = []
    for timestamp_us, node, sensor, values in events:
        results.extend(runtime.push_sample(node, sensor, timestamp_us, values))
    return pd.DataFrame(results)


replay_entry = next(entry for entry in session_entries if int(entry["session"]) == 4)
live_results = replay_recording_as_live(replay_entry)
assert len(live_results) > 0, "Live replay produced no predictions"
display(live_results.head(10))
print(f"Live predictions: {len(live_results)}")
print(f"Haptic requests after debounce/cooldown: {int(live_results['trigger_haptic'].sum())}")

In [ ]:
elapsed = (live_results["timestamp_us"] - live_results["timestamp_us"].min()) / 1_000_000
probability_matrix = np.vstack(live_results["probabilities"].to_numpy())
fig, axis = plt.subplots(figsize=(14, 5))
for class_id, class_name in enumerate(CLASS_NAMES):
    axis.plot(elapsed, probability_matrix[:, class_id], label=class_name)
haptic_times = elapsed[live_results["trigger_haptic"].to_numpy(dtype=bool)]
axis.scatter(haptic_times, np.ones(len(haptic_times)), marker="v", s=60, color="black", label="haptic request")
axis.set(title="Accelerated live replay through exported ONNX model (session 4)", xlabel="Recording time after first prediction (seconds)", ylabel="Probability", ylim=(0, 1.05))
axis.legend(ncols=4)
plt.tight_layout()
plt.show()

## Checks and handoff

The following final gate verifies that all expected artifacts exist. Download
these files from the Colab Files panel or copy the whole `/content/DATASET`
folder to Drive before the runtime disconnects.

In [ ]:
artifact_names = [
    "vantare_bicep_curl_v1.onnx",
    "model_contract.json",
    "metrics.json",
    "feature_names.json",
    "vantare_bicep_curl_pipeline.py",
]
artifact_rows = []
for name in artifact_names:
    path = OUTPUT_DIR / name
    assert path.is_file() and path.stat().st_size > 0, f"Missing output artifact: {path}"
    artifact_rows.append({"artifact": name, "size_bytes": path.stat().st_size, "path": str(path)})
display(pd.DataFrame(artifact_rows))

print("READY: model export, ONNX parity, live preprocessing, and live replay checks passed.")
print("Important: haptic output remains application-level only; this notebook does not modify firmware.")

## Next steps

1. Integrate the saved Python preprocessing module and ONNX model into the
   laptop service receiving webpage sensor packets.
2. Keep haptic output disabled unless the user is actively performing a set.
3. Collect separate sessions for rest/transitions, shoulder swing, fast or
   momentum curls, deliberate sensor remounting, and multiple users.
4. Re-evaluate with complete people and sessions held out before treating the
   system as a general exercise-form classifier.